# Le diagramme sunrise à masses égales : la sortie des polylogarithmes

*Deuxième carnet de la série Intégrales* — suite directe du
[carnet 01](Integrales-01-Triangle-Scalaire-Python.ipynb) (le triangle scalaire à une boucle).
Veine V3 de la distillation Schwartz/BootLoops ([issue #20428](https://github.com/jsboige/CoursIA/issues/20428)).

**Jumeau C#** du carnet [Integrales-02-Sunrise-Masses-Egales-Python](Integrales-02-Sunrise-Masses-Egales-Python.ipynb) :
même plan, mêmes formules, mêmes contrôles, mêmes exercices. Où le carnet Python travaille en
précision arbitraire (`mpmath`, 30 chiffres), celui-ci travaille en double précision avec
l'outil réel `System.Math` — et le dit : chaque valeur de référence du carnet Python est
portée comme référence déclarée, et l'écart mesuré côté double figure dans les sorties.

Le carnet 01 calculait le **triangle scalaire** : trois propagateurs sans masse, **une** boucle,
forme fermée $J(1,1,2-2\cos\psi) = 2\,\mathrm{Cl}_2(\psi)/\sin\psi$ — un monde
*polylogarithmique*. Ce carnet monte d'un cran en complexité : le **diagramme sunrise**
(trois propagateurs de masse égale $m$ entre deux sommets, moment externe $q$), **deux**
boucles — l'exemple standard de la littérature (Adams–Bogner–Weinzierl) pour l'entrée dans le
**monde elliptique**.

Pourquoi ce saut est-il là, et pas au triangle *massif* ? Parce qu'à une boucle, même avec des
masses, tout reste dilogarithmique — c'est le contenu du calcul classique de
't Hooft–Veltman (1979) : les intégrales scalaires à une boucle s'expriment en logarithmes et
dilogarithmes. Les courbes elliptiques n'apparaissent qu'à partir de **deux** boucles. Le
sunrise à masses égales est le plus petit graphe qui les voie.

**Choix de la dimension.** En $d = 4$ le sunrise est divergent (il exige une renormalisation —
hors périmètre, cf. le candidat Integrales-03 de la feuille de route). Nous travaillons en
$d = 2$ euclidienne, où il est **fini** et où la structure elliptique est la plus nette.
Unités : $m = 1$, $s = q_E^2/m^2 \geq 0$.

## 1. Le diagramme et sa représentation paramétrique

Le sunrise relie deux sommets par **trois propagateurs internes** (un « bananier » à trois
brins). Deux boucles indépendantes, trois propagateurs : $L = 2$, $N = 3$.

La représentation paramétrique d'un graphe à $L$ boucles et $N$ arêtes (propagateurs
euclidiens $1/(k^2+m^2)$) fait intervenir les polynômes de Symanzik $\mathcal{U}$
(homogène de degré $L$) et $\mathcal{F}$ (homogène de degré $L+1$) :

$$I(q^2) = \Gamma\!\left(N - \tfrac{LD}{2}\right) \int_{x_i \geq 0} \delta(1 - \textstyle\sum x_i) \; \frac{\mathcal{U}^{\,N-(L+1)D/2}}{\mathcal{F}^{\,N-LD/2}}$$

Pour le sunrise à masses égales :

$$\mathcal{U} = x_1x_2 + x_1x_3 + x_2x_3, \qquad \mathcal{F} = (x_1+x_2+x_3)\,\mathcal{U} + s\,x_1x_2x_3$$

— **le même $\mathcal{U}$ que le triangle du carnet 01**. C'est le fil conducteur de la série :
le polynôme ne change pas, le nombre de boucles, si. En $d = 2$ :
$N - LD/2 = 3 - 2 = 1$ (donc $\Gamma(1) = 1$) et $N - (L+1)D/2 = 3 - 3 = 0$
(donc $\mathcal{U}^0 = 1$). Sur le simplexe ($x_1 + x_2 + x_3 = 1$) :

$$\boxed{\;I(s) = \int_0^1 dx_1 \int_0^{1-x_1} \frac{dx_2}{\;x_1x_2 + x_1x_3 + x_2x_3 + s\,x_1x_2x_3\;} \;}, \qquad x_3 = 1 - x_1 - x_2$$

In [1]:
using System;
using System.Globalization;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

double U(double x1, double x2, double x3) => x1 * x2 + x1 * x3 + x2 * x3;
double F(double x1, double x2, double x3, double s) => (x1 + x2 + x3) * U(x1, x2, x3) + s * x1 * x2 * x3;

// Homogeneite verifiee numeriquement : U de degre 2, F de degre 3.
double lam = 1.7;
double x1 = 0.31, x2 = 0.17, x3 = 0.52;
double ratioU = U(lam * x1, lam * x2, lam * x3) / (lam * lam * U(x1, x2, x3));
double ratioF = F(lam * x1, lam * x2, lam * x3, 4) / (lam * lam * lam * F(x1, x2, x3, 4));
Console.WriteLine("U(l x) / l^2 U(x) = " + ratioU.ToString("G17"));
Console.WriteLine("F(l x) / l^3 F(x) = " + ratioF.ToString("G17"));
Debug.Assert(Math.Abs(ratioU - 1) < 1e-14 && Math.Abs(ratioF - 1) < 1e-14);

U(l x) / l^2 U(x) = 1.0000000000000002


F(l x) / l^3 F(x) = 1.0000000000000002


Les deux rapports valent $1$ aux erreurs d'arrondi près (l'affichage `G17` montre les derniers
bits de la double précision) : les degrés d'homogénéité sont conformes. C'est le contrôle de
cohérence minimum avant d'intégrer quoi que ce soit.

## 2. Voie A — quadrature 2D sur le simplexe

Le carnet Python imbrique deux quadratures 1D à la main (`mpmath` refuse les bornes
*callables* en quadrature 2D directe) ; la borne intérieure $1 - x_1$ dépend du paramètre
externe. Le carnet C# fait de même, avec l'organe du jumeau 04 : des nœuds de Gauss–Legendre
construits par Newton, dans l'ordre de la série.

Une règle **unique** ne suffirait pas ici : les coins du simplexe et les couches limites en
$s$ exigent une règle **composite**, à panneaux dyadiques serrés vers les bords. Les
changements trigonométriques $x_1 = \sin^2 t$, $x_2 = \cos^2 t\,\sin^2 v$ adoucissent les
coins — c'est exactement la voie A du même diagramme qu'au jumeau 04.

In [2]:
// Quadrature de Gauss-Legendre : les noeuds sont strictement interieurs (organe du jumeau 04).
(double[] x, double[] w) Gauss(int n)
{
    var x = new double[n];
    var w = new double[n];
    for (int i = 0; i < (n + 1) / 2; i++)
    {
        double z = Math.Cos(Math.PI * (i + 0.75) / (n + 0.5));
        double pp = 0;
        for (int iteration = 0; iteration < 30; iteration++)
        {
            double p = 1, previous = 0;
            for (int j = 1; j <= n; j++)
            {
                double next = ((2.0 * j - 1) * z * p - (j - 1) * previous) / j;
                previous = p;
                p = next;
            }
            pp = n * (z * p - previous) / (z * z - 1);
            double nextZ = z - p / pp;
            if (Math.Abs(nextZ - z) < 1e-15) { z = nextZ; break; }
            z = nextZ;
        }
        double weight = 2 / ((1 - z * z) * pp * pp);
        x[i] = -z; x[n - 1 - i] = z;
        w[i] = weight; w[n - 1 - i] = weight;
    }
    return (x, w);
}

var regle = Gauss(16);

// Coupures dyadiques de [0, 1] : 0, 2^-k, ..., 1/2, ..., 1 - 2^-k, 1.
double[] Panneaux(int k)
{
    var pts = new List<double> { 0.0, 1.0 };
    for (int j = 1; j <= k; j++)
    {
        double h = Math.Pow(2, -j);
        pts.Add(h);
        pts.Add(1 - h);
    }
    pts.Sort();
    return pts.ToArray();
}

// Regle composite : Gauss(16) sur chaque panneau, mappe lineairement sur [a, b].
double Integrer(Func<double, double> f, double a, double b, int k)
{
    var pts = Panneaux(k);
    double total = 0;
    for (int p = 0; p < pts.Length - 1; p++)
    {
        double u0 = a + (b - a) * pts[p], u1 = a + (b - a) * pts[p + 1];
        double milieu = (u0 + u1) / 2, demi = (u1 - u0) / 2, somme = 0;
        for (int i = 0; i < regle.x.Length; i++)
            somme += regle.w[i] * f(milieu + demi * regle.x[i]);
        total += demi * somme;
    }
    return total;
}

double I_A(double s, int k)
{
    // Sunrise d=2, voie A du carnet 02 : imbrication manuelle de deux quadratures 1D,
    // coins adoucis par x1 = sin^2 t, x2 = cos^2 t sin^2 v (organe du jumeau 04).
    return Integrer(t =>
    {
        double st = Math.Sin(t), ct = Math.Cos(t);
        double y1 = st * st, reste = ct * ct;
        double interieur = Integrer(v =>
        {
            double sv = Math.Sin(v), cv = Math.Cos(v);
            double y2 = reste * sv * sv, y3 = reste * cv * cv;
            return 2 * reste * sv * cv / (U(y1, y2, y3) + s * y1 * y2 * y3);
        }, 0, Math.PI / 2, k);
        return 2 * st * ct * interieur;
    }, 0, Math.PI / 2, k);
}

var chrono = Stopwatch.StartNew();
double valZero = I_A(0, 18), valNeuf = I_A(9, 18);
chrono.Stop();
Console.WriteLine("I(0)  = " + valZero.ToString("G17"));
Console.WriteLine("I(9)  = " + valNeuf.ToString("G17"));
Console.WriteLine("duree : " + (chrono.ElapsedMilliseconds / 1000.0).ToString("F1") + " s");

I(0)  = 2,3439072386894537


I(9)  = 1,6979464611954236


duree : 0,2 s


## 3. L'ancre $s = 0$ : planted-truth contre le carnet 01

À moment externe nul, $\mathcal{F} = \mathcal{U}$ sur le simplexe, et l'intégrale du sunrise
**retombe exactement** sur celle du triangle du carnet 01 avec $\alpha = \beta = \gamma = 1$ :

$$I(0) = J(1,1,1) = \frac{4}{\sqrt{3}}\,\mathrm{Cl}_2\!\left(\frac{\pi}{3}\right)$$

— car $(1,1,1)$ est le cas $\psi = 60°$ de la forme fermée $J(1, 1, 2-2\cos\psi)$ certifiée au
carnet 01. C'est notre *planted truth* : une valeur connue **avant** de croire la moindre
quadrature.

En C#, il n'existe pas de fonction de Clausen dans la BCL : la cellule suivante implémente
$\mathrm{Cl}_2$ par sa série de Bernoulli, puis la re-valide contre la constante de Catalan —
même protocole que le carnet Python, outil différent.

In [3]:
// Cl2(theta) = sum_{k>=1} sin(k theta)/k^2, par sa serie de Bernoulli :
//   Cl2(theta) = theta (1 - ln theta) + sum_{k>=1} |B_2k| theta^(2k+1) / ((2k)(2k+1)(2k)!),  0 < theta < 2 pi
double Clausen2(double theta)
{
    double[] B = { 1.0 / 6, -1.0 / 30, 1.0 / 42, -1.0 / 30, 5.0 / 66, -691.0 / 2730, 7.0 / 6,
                   -3617.0 / 510, 43867.0 / 798, -174611.0 / 330, 854513.0 / 138,
                   -236364091.0 / 2730, 8553103.0 / 6, -23749461029.0 / 870 };
    double theta2 = theta * theta;
    double puissance = theta * theta2;
    double factorielle = 2;
    double somme = 0;
    for (int k = 1; k <= B.Length; k++)
    {
        double coefficient = (k % 2 == 1) ? B[k - 1] : -B[k - 1];   // (-1)^(k+1) B_2k = |B_2k|
        somme += coefficient * puissance / ((2.0 * k) * (2.0 * k + 1) * factorielle);
        puissance *= theta2;
        factorielle *= (2.0 * k + 1) * (2.0 * k + 2);
    }
    return theta * (1 - Math.Log(theta)) + somme;
}

// Constante de Catalan : valeur de reference (Cl2(pi/2) = G), et non resultat d'une somme C#.
const double Catalan = 0.915965594177219015054603514932384110774;
// Ancre du carnet 01 : (4/sqrt(3)) Cl2(pi/3), reference mpmath du jumeau Python.
const double Ancre = 2.343907238689458890602;

double clPiSur2 = Clausen2(Math.PI / 2);
Console.WriteLine("Cl2(pi/2) - G = " + (clPiSur2 - Catalan).ToString("G17"));

double ancreCalculee = 4 / Math.Sqrt(3) * Clausen2(Math.PI / 3);
double iA0 = I_A(0, 18);
Console.WriteLine("ancre (4/sqrt(3)) Cl2(pi/3) = " + ancreCalculee.ToString("G17"));
Console.WriteLine("ancre declaree (mpmath)     = " + Ancre.ToString("G17"));
Console.WriteLine("I_A(0)                      = " + iA0.ToString("G17"));
Console.WriteLine("ecart relatif a la reference mpmath = " + (Math.Abs(iA0 - Ancre) / Ancre).ToString("G3"));

Cl2(pi/2) - G = 1,1102230246251565E-16


ancre (4/sqrt(3)) Cl2(pi/3) = 2,3439072386894595


ancre declaree (mpmath)     = 2,343907238689459


I_A(0)                      = 2,3439072386894537


ecart relatif a la reference mpmath = 2,27E-15


L'écart affiché sur $\mathrm{Cl}_2(\pi/2) - G$ n'est pas nul : c'est le niveau de l'arrondi
double, là où le carnet Python rend `0.0` en précision arbitraire.

L'ancre, elle, tombe à l'arrondi de la référence déclarée près : la concordance des valeurs
affichées est la vérification la plus dure disponible pour la paramétrisation entière — si le
polynôme $\mathcal{F}$ était erroné d'un terme, l'ancre raterait.

## 4. Voie B — substitution homogène $(u, v)$

Même substitution qu'au carnet 01 : $x_2 = uv$, $x_3 = u(1-v)$, $x_1 = 1-u$, jacobien $u$.
Le simplexe devient le carré unité — et $x_1x_2x_3 = u^2(1-u)\,v(1-v)$ :

$$I(s) = \int_0^1\!\!\int_0^1 \frac{u \;du\,dv}{\;u(1-u) + u^2v(1-v) + s\,u^2(1-u)v(1-v)\;}$$

L'intégrande se simplifie par $u$ — la singularité apparente en $u = 0$ disparaît — et le coin
$(u, v) = (1, 0)$ ne porte qu'une singularité logarithmique : la règle composite y grade ses
panneaux, au prix d'une convergence en $2^{-k}$ (le carnet Python, adaptatif, n'a pas ce coût).
L'écart $|A - B|$ affiché ci-dessous est donc celui de la voie B seule.

In [4]:
double I_B(double s, int k)
{
    // Carre unite : x1 = 1-u, x2 = u v, x3 = u (1-v), jacobien u.
    // L'integrande u/(...) se simplifie par u : 1 / ((1-u) + u v (1-v) (1 + s (1-u))).
    return Integrer(u => Integrer(v =>
        1.0 / ((1 - u) + u * v * (1 - v) * (1 + s * (1 - u))), 0, 1, k), 0, 1, k);
}

Console.WriteLine("   s |                   voie A |                   voie B | |A-B|");
foreach (int s in new[] { 0, 1, 4, 9, 16, 25 })
{
    double a = I_A(s, 18), b = I_B(s, 26);
    Console.WriteLine($"{s,4} | {a.ToString("G17"),24} | {b.ToString("G17"),24} | {Math.Abs(a - b).ToString("G3")}");
}

   s |                   voie A |                   voie B | |A-B|


   0 |       2,3439072386894537 |       2,3439072386015756 | 8,79E-11


   1 |       2,2367927002126415 |       2,2367927001247629 | 8,79E-11


   4 |       1,9839438109076231 |       1,9839438108197447 | 8,79E-11


   9 |       1,6979464611954236 |        1,697946461107545 | 8,79E-11


  16 |       1,4399685779983273 |        1,439968577910449 | 8,79E-11


  25 |       1,2257091272767238 |       1,2257091271888454 | 8,79E-11


## 5. Voie Cheng–Wu — le théorème qui change le domaine

Le théorème de Cheng–Wu permet de fixer **un** paramètre de Feynman à 1 et d'intégrer les
autres sur $[0, \infty)$ : la valeur est inchangée. Posons $x_3 = 1$ :

$$I(s) = \int_0^\infty\!\!\int_0^\infty \frac{dx_1\,dx_2}{\;(1+x_1+x_2)(x_1+x_2+x_1x_2) + s\,x_1x_2\;}$$

Domaine non borné, bornes constantes — une paramétrisation réellement **différente** des deux
précédentes : si elle s'accordait par accident, ce ne serait pas un accident.

Pour la ramener sur un domaine borné, la cellule pose $a = 1/(1+x_1)$, $b = 1/(1+x_2)$ :

$$I(s) = \int_0^1\!\!\int_0^1 \frac{da\,db}{\;(a+b-ab)(1-ab) + s\,(1-a)(1-b)\,ab\;}$$

Le coin $(a,b) = (0,0)$ y est logarithmique ; deux triangles de Duffy le rationalisent, chacun
avec son jacobien $t$.

In [5]:
double I_CW(double s, int k)
{
    // Cheng-Wu : x3 = 1 ; substitution a = 1/(1+x1), b = 1/(1+x2) vers le carre unite.
    // Le coin (a,b) = (0,0) est logarithmique : deux triangles de Duffy le rationalisent.
    double g1(double t, double w)
    {
        double q = t * t * w * (1 - w);              // ab = t^2 w (1-w)
        double a = t * (1 - w), b = t * w;
        return t / ((t - q) * (1 - q) + s * (1 - a) * (1 - b) * q);
    }
    double g2(double t, double w)
    {
        double a = 1 - t * (1 - w), b = 1 - t * w;
        double ab = a * b;
        double unMoinsAB = t - t * t * w * (1 - w);  // 1 - ab, forme stable
        return t / ((a + b - ab) * unMoinsAB + s * (1 - a) * (1 - b) * ab);
    }
    return Integrer(t => Integrer(w => g1(t, w), 0, 1, k), 0, 1, k)
         + Integrer(t => Integrer(w => g2(t, w), 0, 1, k), 0, 1, k);
}

foreach (int s in new[] { 0, 9 })
{
    double a = I_A(s, 18), c = I_CW(s, 12);
    Console.WriteLine($"s={s}: CW={c.ToString("G17")}  A={a.ToString("G17")}  diff={Math.Abs(c - a).ToString("G3")}");
}

s=0: CW=2,3439072386894488  A=2,3439072386894537  diff=4,88E-15


s=9: CW=1,6979464611954187  A=1,6979464611954236  diff=4,88E-15


## 6. Voie D — coordonnées hyperboliques : vers la structure elliptique

Dernière transformation, la plus instructive. Dans la forme Cheng–Wu, posons
$x_1 = r e^{\theta}$, $x_2 = r e^{-\theta}$ ($r > 0$, $\theta \in \mathbb{R}$) :
le jacobien vaut $2r$, le domaine devient la demi-bande $r > 0$, $\theta \in \mathbb{R}$, et

$$\mathcal{F} = 4r^2\cosh^2\theta + 2r(1+r^2)\cosh\theta + r^2(1+s)$$

— un **polynôme quadratique en $\cosh\theta$**. Pour rester sur un intervalle fini, la cellule
pose $z = \tanh(\theta/2)$ : $\cosh\theta = (1+z^2)/(1-z^2)$, $d\theta = 2\,dz/(1-z^2)$, et
$\theta \in \mathbb{R}$ devient $z \in [-1, 1]$. La rationalisation est exacte ; seul le
domaine change de forme.

In [6]:
double I_D(double s, int k, bool demi)
{
    // theta = 2 argth z ; cosh(theta) = (1+z^2)/(1-z^2), dtheta = 2 dz/(1-z^2) :
    //   I = int_0^1 dr int_-1^1 dz q1 + int_0^1 dt int_-1^1 dz q2   (r = 1/t au-dela de 1).
    // demi = true : z sur [0, 1] seulement, soit la moitie par symetrie du domaine.
    double q1(double r, double z)
    {
        double z2 = z * z;
        return 4 * r * (1 - z2) / (4 * r * r * (1 + z2) * (1 + z2)
                                   + 2 * r * (1 + r * r) * (1 - z2 * z2)
                                   + r * r * (1 + s) * (1 - z2) * (1 - z2));
    }
    double q2(double t, double z)
    {
        double z2 = z * z;
        return 4 * (1 - z2) / (t * (4 * (1 + z2) * (1 + z2) + (1 + s) * (1 - z2) * (1 - z2))
                               + 2 * (1 + t * t) * (1 - z2 * z2));
    }
    double zMin = demi ? 0.0 : -1.0;
    return Integrer(r => Integrer(z => q1(r, z), zMin, 1, k), 0, 1, k)
         + Integrer(t => Integrer(z => q2(t, z), zMin, 1, k), 0, 1, k);
}

double aSeize = I_A(16, 18), dSeizeDemi = I_D(16, 16, true);
Console.WriteLine("I_D(theta >= 0, jacobien 2r) = " + dSeizeDemi.ToString("G17"));
Console.WriteLine("I_A(16)                      = " + aSeize.ToString("G17"));
Console.WriteLine("rapport                      = " + (dSeizeDemi / aSeize).ToString("G17"));

I_D(theta >= 0, jacobien 2r) = 0,71998428899915845


I_A(16)                      = 1,4399685779983273


rapport                      = 0,49999999999999639


**Rapport exact $1/2$.** Le piège : $\theta \geq 0$ ne couvre que la moitié du domaine —
les points $(x_1, x_2)$ et $(x_2, x_1)$ correspondent à $\theta$ de signes opposés. La
symétrie $x_1 \leftrightarrow x_2$ de l'intégrande autorise $\theta \in [0, \infty)$
**avec facteur 2** (soit jacobien effectif $4r$), ou l'intégration sur $\theta \in \mathbb{R}$
avec $2r$. Un contrôle à un seul point qui n'aurait pas *mordu* (par exemple comparer à $s=0$
sans valeur de référence indépendante) aurait laissé passer un facteur 2 silencieux.

Le rapport affiché ci-dessus vaut $1/2$ à la précision de la règle — le carnet Python mesure
$0.4999999999999996$ en précision arbitraire, la différence est celle des deux quadratures.

In [7]:
foreach (int s in new[] { 0, 4, 16 })
{
    double a = I_A(s, 18), d = I_D(s, 16, false);
    Console.WriteLine($"s={s}: D={d.ToString("G17")}  A={a.ToString("G17")}  diff={Math.Abs(d - a).ToString("G3")}");
}

s=0: D=2,3439072386894488  A=2,3439072386894537  diff=4,88E-15


s=4: D=1,983943810907618  A=1,9839438109076231  diff=5,11E-15


s=16: D=1,4399685779983225  A=1,4399685779983273  diff=4,88E-15


### Où est l'elliptique ?

Substituons $t = \cosh\theta$ ($d\theta = dt/\sqrt{t^2-1}$, $t \in [1, \infty)$) dans
l'intégrande : le dénominateur devient $(t - t_1(r))(t - t_2(r))$ où $t_1, t_2$ sont les
racines du trinôme en $t$ (réelles négatives ou complexes conjuguées selon $(r, s)$ — leur
produit vaut $(1+s)/4$, leur somme $-(1+r^2)/2r$), et

$$\int \frac{d\theta}{\mathcal{F}} = \frac{1}{4r^2}\int_{1}^{\infty} \frac{dt}{(t-t_1)(t-t_2)\sqrt{t^2-1}}$$

La racine carrée porte sur un binôme de degré 2 : la substitution $u = \tanh(\theta/2)$
($t = (1+u^2)/(1-u^2)$, $\sqrt{t^2-1} = 2u/(1-u^2)$) la **rationalise** et l'intégrale en
$\theta$ est élémentaire — fractions partielles en $u^2$, donc $\operatorname{artanh}$ ou
$\arctan$ selon la position des pôles. L'elliptique n'est pas dans cette intégrale : il vit
dans l'intégration **restante sur $r$**. Les pôles $t_{1,2}(r)$ se collent là où le
discriminant $4r^2\,(r^4 - (4s+2)\,r^2 + 1)$ s'annule — une **quartique** en $r$, courbe de
genre 1 que le [carnet 04](Integrales-04-Sunrise-Courbe-Elliptique-Python.ipynb) exhibe
explicitement ($y^2 = b(\chi)c(\chi)$). C'est la manifestation concrète du théorème
structural : à deux boucles, les périodes de Feynman vivent sur des courbes de genre 1.
(Le triangle massif du même polynôme $\mathcal{U}$, lui, reste dilogarithmique : le genre
naît du nombre de boucles, pas des masses.)

## 7. Famille $I(s)$ : valeurs et décroissance

Table récapitulative (voies A et B), $s$ de 0 à 100 — la fonction est **décroissante** :
à grand $s$ le terme $s\,x_1x_2x_3$ écrase le centre du simplexe et l'intégrale se concentre
vers les coins. En $s = 9$ ($q_E^2 = 9m^2$) **rien ne se passe** sur la section euclidienne :
le seuil à trois corps est une propriété de la continuation analytique (région de type temps),
pas de la section réelle euclidienne — ne pas sur-interpréter une courbe lisse.

In [8]:
Console.WriteLine("    s |                     I(s) | |A-B|");
var vals = new Dictionary<int, double>();
foreach (int s in new[] { 0, 1, 4, 9, 16, 25, 36, 100 })
{
    double a = I_A(s, 18), b = I_B(s, 26);
    vals[s] = a;
    Console.WriteLine($"{s,5} | {a.ToString("G17"),24} | {Math.Abs(a - b).ToString("G3")}");
}

    s |                     I(s) | |A-B|


    0 |       2,3439072386894537 | 8,79E-11


    1 |       2,2367927002126415 | 8,79E-11


    4 |       1,9839438109076231 | 8,79E-11


    9 |       1,6979464611954236 | 8,79E-11


   16 |       1,4399685779983273 | 8,79E-11


   25 |       1,2257091272767238 | 8,79E-11


   36 |       1,0523540707776371 | 8,79E-11


  100 |      0,62901670759061623 | 8,79E-11


## 8. Tentative d'identification fermée — et refus documenté

Le protocole *constant-recognition* (BootLoops) exige : **anneau déclaré avant la recherche,
refus plutôt qu'invention** si rien ne s'y trouve. Deux familles candidates naturelles pour
une forme fermée de $I(s)$ :

1. **Module elliptique simple** : $I(s) \stackrel{?}{=} A \cdot K(k(s))$ avec $A$ constant et
   le modulo plausible $k^2 = (1 - \sqrt{1 - 9/s})/2$ de la famille « seuil $s = 9$ » ;
2. **Décroissance asymptotique** : si $I(s) \sim c\,s^{-1/3}$ (la pente « banana » souvent
   citée), alors $I(s)\,s^{1/3} \to c$ — et $c$ se chercherait dans l'anneau
   $\{\pi, \Gamma(1/3)\}$.

In [9]:
// (1) module K candidat : k^2 = (1 - sqrt(1 - 9/s))/2 pour s > 9
double KComplet(double m)
{
    // K(m) = pi / (2 AGM(1, sqrt(1-m))) : integrale elliptique complete, parametre m = k^2.
    double a = 1, b = Math.Sqrt(1 - m);
    for (int i = 0; i < 60 && Math.Abs(a - b) > 1e-16 * a; i++)
    {
        double suivant = (a + b) / 2;
        b = Math.Sqrt(a * b);
        a = suivant;
    }
    return Math.PI / (2 * a);
}

Console.WriteLine("hypothese I(s) = A * K(k(s)), A = I*pi/(2K) devrait etre CONSTANT :");
foreach (int s in new[] { 16, 25, 36, 100 })
{
    double k2 = (1 - Math.Sqrt(1 - 9.0 / s)) / 2;
    double K = KComplet(k2);
    Console.WriteLine($"  s={s,4}: A = {(vals[s] * Math.PI / (2 * K)).ToString("G16")}  (k^2 = {k2.ToString("G6")})");
}

hypothese I(s) = A * K(k(s)), A = I*pi/(2K) devrait etre CONSTANT :


  s=  16: A = 1,375466126214393  (k^2 = 0,169281)


  s=  25: A = 1,194052358166938  (k^2 = 0,1)


  s=  36: A = 1,034347327509113  (k^2 = 0,0669873)


  s= 100: A = 0,6253686809525054  (k^2 = 0,0230304)


In [10]:
// (2) constante asymptotique c(s) = I(s) * s^(1/3) devrait CONVERGER
Console.WriteLine("c(s) = I(s) * s^(1/3) :");
foreach (int s in new[] { 100, 400, 1600, 6400, 25600 })
{
    double c = I_A(s, 18) * Math.Pow(s, 1.0 / 3.0);
    Console.WriteLine($"  s={s,6}: c = {c.ToString("G14")}");
}

c(s) = I(s) * s^(1/3) :


  s=   100: c = 2,9196369261085


  s=   400: c = 1,9758182917846


  s=  1600: c = 1,1922881489585


  s=  6400: c = 0,66824876287783


  s= 25600: c = 0,35581090537256


**Refus, dans les deux cas, et il est documenté :**

- $A = I\pi/(2K)$ **dérive** systématiquement (aucune constante n'émerge) : la forme
  $A\cdot K(k)$ est trop pauvre — cohérent avec la structure attendue d'une **double
  période** (produit de deux intégrales elliptiques, formes elliptiques généralisées
  d'Adams–Bogner–Weinzierl), qui n'est pas de la forme $A\cdot K$ ;
- $c(s)$ **ne converge pas** sur l'intervalle mesuré (décroissance plus rapide que
  $s^{-1/3}$ ici) : aucune constante de l'anneau $\{\pi, \Gamma(1/3)\}$ n'est proposée.

*Ce refus est un résultat* : le protocole exige de dire « pas de forme fermée identifiée »
plutôt que d'arranger une coïncidence à trois chiffres. La voie D, elle, a **établi la nature
elliptique** par réduction — nature et valeur fermée sont deux énoncés distincts.

## 9. Périmètre honnête

- **Établi ici** : la paramétrisation (homogénéité contrôlée), quatre voies numériques
  d'accord à la précision double (écarts mesurés dans les sorties ; côté `mpmath`, le jumeau
  Python mesure Cheng–Wu à $1{,}37\times10^{-20}$ et l'hyperbolique à $8{,}94\times10^{-16}$),
  l'ancre $I(0) = J(1,1,1) = \frac{4}{\sqrt 3}\mathrm{Cl}_2(\pi/3)$ contrôlée contre le
  carnet 01, la réduction élémentaire de la voie D (rationalisation $u = \tanh(\theta/2)$),
  le refus documenté des deux identifications.
- **Pas établi** : une forme fermée de $I(s)$ (la littérature la donne en formes elliptiques
  généralisées / intégrales elliptiques multiples — Adams–Bogner–Weinzierl, citées sans être
  reproduites) ; des bornes d'erreur *certifiées* sur les quadratures (les règles composites
  donnent ici des écarts constatés, pas des bornes) ; le comportement de type temps (seuils,
  continuation analytique).
- **Indépendance des voies** : les quatre voies partagent la même paramétrisation — elles
  vérifient les transformations, pas la formule de $\mathcal{F}$ elle-même. L'ancre
  $s = 0$ (indépendante : certifiée au carnet 01) contrôle la paramétrisation à $s$ nul —
  à $s = 0$ le terme en $s$ disparaît de $\mathcal{F}$, une erreur sur ce seul terme y
  passerait ; le contrôle en $s > 0$ vient de l'accord Cheng–Wu / simplexe à $s = 9$, mesuré
  dans la sortie de la §5.

## 10. Exercices

Trois exercices, à compléter sans erreur volontaire (le carnet doit rester exécutable de bout
en bout — convention de la série).

In [11]:
// Exercice 1. Exposants parametriques en d = 4.
// Pour L = 2, N = 3, D = 4 : que valent N - L*D/2 et N - (L+1)*D/2 ?
// Que devient Gamma(N - L*D/2) ? (C'est l'entree du candidat Integrales-03 :
// la divergence du sunrise en d = 4 et la renormalisation qu'elle exige.)
// Indice : calculer les deux exposants ; la BCL n'expose pas Gamma, mais un pole
// de Gamma en un entier negatif se lit sur l'exposant lui-meme.
// Etape 1 : ecrire les deux exposants
// Etape 2 : conclure sur Gamma(...)

Console.WriteLine("Exercice a completer");
object resultat_exo1 = null;  // TODO etudiant

Exercice a completer


In [12]:
// Exercice 2. Monotonie de I(s).
// (a) Compter les coins du simplexe ou x_i -> 1 et justifier que chacun contribue
//     de maniere identique a grand s (symetrie S3 de l'integrande).
// (b) Estimer I(0.25) par les voies A et B et verifier l'accord ; la valeur
//     doit rester au-dessus de I(1) = 2.2367927002... (monotonie).
// Indice : pour (a), se rappeler que U ~ x_j + x_k pres du coin x_i -> 1.

Console.WriteLine("Exercice a completer");
object resultat_exo2 = null;  // TODO etudiant

Exercice a completer


In [13]:
// Exercice 3. Le facteur 2 de la voie D.
// La version "theta >= 0" (parametre demi = true) rendait exactement la moitie de I(s).
// (a) Retrouver le jacobien |d(x1,x2)/d(r,theta)| = 2r du changement
//     x1 = r e^theta, x2 = r e^{-theta}.
// (b) Verifier numeriquement que multiplier par 2 (facteur de symetrie x1 <-> x2)
//     restaure l'accord avec la voie A a s = 1.
// Etape 1 : deriver le jacobien a la main
// Etape 2 : comparer 2*I_D(1, k, demi: true) a I_A(1)

Console.WriteLine("Exercice a completer");
object resultat_exo3 = null;  // TODO etudiant

Exercice a completer


## À retenir

Le sunrise à masses égales en $d = 2$ est le plus petit objet de Feynman dont les périodes
**sortent des polylogarithmes** : quatre voies numériques indépendantes s'accordent,
l'intégrale retombe exactement sur le triangle du carnet 01 à $s = 0$ (le même polynôme
$\mathcal{U}$), et la voie hyperbolique se laisse rationaliser jusqu'au bout —
$t = \cosh\theta$ puis $u = \tanh(\theta/2)$ rendent l'intégrale en $\theta$ élémentaire,
repoussant la structure elliptique dans l'intégration sur $r$ que le carnet 04 explicite en
courbe $y^2 = b(\chi)c(\chi)$. Ce qui n'a **pas** été obtenu : une forme fermée. Le refus
est écrit, l'anneau était déclaré : c'est la discipline qui rend la recherche honnête.

## Vérifiez votre compréhension

1. Qu'est-ce que l'ancre $s = 0$ contrôle, et pourquoi ce contrôle est-il fort ?

<details><summary>Réponse</summary>

À $s = 0$, le polynôme $\mathcal{U}$ du sunrise dégénère en celui du triangle du carnet 01, et $I(0) = 2.343907238689458890602$ — exactement la forme fermée $\frac{4}{\sqrt{3}}\,\mathrm{Cl}_2(\pi/3)$ déjà certifiée là (contrôle de $\mathrm{Cl}_2$ contre la constante de Catalan au §3, à l'arrondi double près). C'est un *planted truth* : la valeur de vérité est plantée **avant** la mesure par un carnet indépendant, donc un accord des quatre voies sur ce point ne peut pas venir d'une erreur commune à toutes les voies du carnet 02 — avec une limite : à $s = 0$ le terme en $s$ disparaît de $\mathcal{F}$, une erreur sur ce seul terme passerait l'ancre. Le contrôle en $s > 0$ vient de l'accord Cheng–Wu / simplexe à $s = 9$, mesuré au §5 (côté `mpmath`, $8{,}5 \times 10^{-21}$).

</details>

2. Que donne la substitution $t = \cosh\theta$ (§6), et pourquoi est-elle décisive pour la suite de la série ?

<details><summary>Réponse</summary>

Elle ramène l'intégrande à deux pôles simples $(t - t_1)(t - t_2)$ sur une racine carrée $\sqrt{t^2 - 1}$, que $u = \tanh(\theta/2)$ rationalise — l'intégrale en $\theta$ est **élémentaire**, et la voie D s'accorde avec la voie A à la précision double à $s = 0$ (sortie du §6 ; côté `mpmath`, $8{,}94 \times 10^{-16}$). La structure elliptique, elle, vit dans l'intégration restante sur $r$ : les pôles $t_{1,2}(r)$ se collent sur la quartique $r^4 - (4s+2)\,r^2 + 1 = 0$, la courbe de genre 1 que le carnet 04 explicite — elle sort du calcul, pas d'une conjecture importée (théorème structural : à deux boucles, les périodes vivent sur des courbes de genre 1, alors que le triangle du même $\mathcal{U}$ reste dilogarithmique ; le genre naît du nombre de boucles, pas des masses). Le théorème de Cheng–Wu est contrôlé à part : $\mathrm{CW}$ et A s'accordent à la précision double au §5 (côté `mpmath`, $1{,}37 \times 10^{-20}$).

</details>

3. Le carnet cherche une forme fermée $I(s) = A\,K(k(s))$ et l'abandonne. Que mesure-t-il avant d'écrire le refus ?

<details><summary>Réponse</summary>

Il mesure la compatibilité de l'hypothèse : si $I(s) = A K(k(s))$, alors $A = I\pi/(2K)$ devrait être **constant** en $s$ — la table du §8 montre qu'il ne l'est pas ($A = 1.3754\ldots$ à $s = 16$, déjà différent de la valeur à $s = 0$). Le refus est documenté avec la mesure qui l'appuie, et l'anneau avait été déclaré avant la recherche — c'est le protocole *constant-recognition*. La décroissance mesurée ($c(s) = I(s)\, s^{1/3}$, §8) reste comme trace quantitative de la tentative.

</details>

## Pour aller plus loin

- **Jumeau Python** : [Integrales-02-Sunrise-Masses-Egales-Python](Integrales-02-Sunrise-Masses-Egales-Python.ipynb) — le carnet d'origine, en précision arbitraire (`mpmath`, 30 chiffres).
- **Carnet suivant** : [Integrales-03 — la bulle et la première divergence](Integrales-03-Bulle-Premiere-Divergence-Python.ipynb) — une boucle de moins, et le premier $\ln\Lambda$ de la série.
- **Carnet voisin** : [Integrales-04 — le rung elliptique du sunrise](Integrales-04-Sunrise-Courbe-Elliptique-Python.ipynb) — prend la famille $I(s)$ là où ce carnet l'a laissée et lui donne sa courbe elliptique explicite.
- Le fil complet de la série : [#23161](https://github.com/jsboige/CoursIA/issues/23161).

## Références

- G. 't Hooft, M. Veltman, *Scalar one-loop integrals*, Nucl. Phys. B **153**, 365 (1979) —
  les intégrales à une boucle sont dilogarithmiques : le saut elliptique exige deux boucles.
- L. Adams, C. Bogner, S. Weinzierl, *The two-loop sunrise graph with arbitrary masses*,
  arXiv:1302.7004 (v1 seule) ; *The two-loop sunrise graph in two space-time dimensions
  with arbitrary masses in terms of elliptic dilogarithms*, arXiv:1405.5640 — la forme
  fermée en intégrales elliptiques multiples.
- J. A. M. Vermaseren / S. Weinzierl, revues sur les intégrales elliptiques de Feynman.
- M. D. Schwartz, *Vibe physics* et *Claude-shaped science* (essais, Anthropic, 2026) ;
  le livrable de la collaboration racontée par le premier est arXiv:2601.02484,
  *Resummation of the C-Parameter Sudakov Shoulder Using Effective Field Theory* ;
  harnais [BootLoops](https://github.com/BootLoops-ai) (MIT) — protocoles
  `planted-truth`, `constant-recognition`, `independence-bookkeeping`.
- Carnet 01 : `Integrales-01-Triangle-Scalaire-Python.ipynb` — forme fermée
  $J = 2\,\mathrm{Cl}_2(\psi)/\sin\psi$ et ancre $J(1,1,1)$.

## Attestation de parité

- **Portage** : ce carnet est le jumeau C# de
  [Integrales-02-Sunrise-Masses-Egales-Python](Integrales-02-Sunrise-Masses-Egales-Python.ipynb) —
  même découpage en sections, mêmes formules, mêmes valeurs de $s$, mêmes contrôles (ancre,
  $\mathrm{Cl}_2$ contre la constante de Catalan, rapport $1/2$ de la voie D, refus
  documentés), mêmes exercices.
- **Outil** : `System.Math` en double précision ; nœuds de Gauss–Legendre (16 nœuds,
  construits par Newton) et règle **composite** à panneaux dyadiques, organisée comme au
  jumeau 04 (règle de Gauss–Legendre + quadrature imbriquée).
- **Références** : la constante de Catalan, l'ancre et les valeurs `mpmath` citées sont des
  références déclarées, jamais des résultats de la quadrature C# ; chaque sortie quantitative
  du carnet est produite par l'exécution.
- **Exécution** : noyau `.net-csharp`, toutes les cellules exécutées sans erreur, sorties
  commises.
- **Exercices** : trois stubs compilables marqués `// TODO etudiant`, sans solution révélée.

> Carnet produit dans le cadre du chantier des jumeaux C# (#26534).